# Проект 1. Аналитическая записка

Автор: Ирина Чернова (idchernova)

## 0. Бизнес-вопрос и гипотезы

1 марта 2026 года «Базовый» подорожал с 199 до 249 руб (+25%), «Расширенный» — с 399 до 449 руб (+12,5 %). После этого выручка заметно выросла.

**Вопрос:** сколько выручки в месяц принесло само подорожание и не потеряли ли мы из-за него подписчиков?

Гипотезы:
- 1. Рост выручки после 1 марта дала в основном цена, а не рост числа подписчиков.
- 2. Средний чек вырос ещё и потому, что клиенты перешли на «Расширенный».
- 3. Подорожание ускорило отток.
- 4. Изменение выручки дал один канал привлечения.

Ключевая метрика: выручка от подписок в день, пересчитанная в нормальный месяц. Сравниваю январь-февраль 2026 с мартом–июлем 2026.

In [ ]:
import itertools

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sqlalchemy import text

from prime.config import get_engine

engine = get_engine()


def q(sql: str, **params) -> pd.DataFrame:
    """Выполнить запрос и вернуть DataFrame."""
    try:
        with engine.connect() as conn:
            conn.execute(text("set time zone 'UTC'"))  # сутки и месяцы по UTC
            return pd.read_sql(text(sql), conn, params=params)
    finally:
        engine.dispose()


def rub(x: float) -> str:
    return f"{x:,.0f} руб".replace(",", " ")


print("подключились как:", q("select current_user as login").iloc[0, 0])

## 1. Данные и зерно расчёта

### 1.1 Выручка по месяцам

Выручка по месяцам как есть и в пересчёте на день. Делю на число дней, в которые в данных есть платежи (days_obs).

In [ ]:
overview = q("""
    select to_char(paid_at, 'YYYY-MM')        as month,
           count(*)                           as n_payments,
           count(distinct paid_at::date)      as days_obs,
           sum(amount)                        as revenue
    from prime.payments
    where status = 'success'
    group by 1
    order by 1
""")

overview["days_calendar"] = pd.PeriodIndex(overview["month"], freq="M").days_in_month
overview["revenue_per_day"] = overview["revenue"] / overview["days_obs"]
overview["growth_pct"] = overview["revenue"].pct_change() * 100                  
overview["growth_per_day_pct"] = overview["revenue_per_day"].pct_change() * 100   

overview.round(1)

сначала:из базы берутся все успешные платежи и группируются по месяцам. Для каждого месяца считается число платежей, выручка и сколько дней в месяце были платежи. Потом добавляется, сколько дней в месяце по календарю, выручка в день (выручка/дни с платежами) и прирост к прошлому месяцу двумя способами (как есть и в пересчёте на день).

для чего: месяцы разной длины, а подписка списывается раз в 30 дней, поэтому выручку месяцев напрямую сравнивать нельзя. К тому же в апреле 2026 нет данных за 14–15 число.

Что показывает: до марта 2026 выручка в день росла ровно, примерно на 3% в месяц. В марте скачок примерно на 21% - в тот же момент, когда подняли цены. После марта рост снова ровный, но медленнее, около 2% в месяц.

Падения в феврале и апреле в колонке growth_pct ненастоящие: февраль короче на три дня, а в апреле нет данных за 14–15 число. В пересчете на день их нет. Январь 2025 (данные с 6-го) и август 2026 неполные, в сравнение их не беру.

Поэтому дальше сравнивается выручка в день, а не за месяц. Единственное настоящее изменение - мартовский скачок.

1.2 Зерно, период, сегмент

Одна строка таблицы monthly: месяц - тариф - канал привлечения: активные подписки, успешные платежи и выручка.

Почему такое зерно. Выручка считается по платежам: каждый платеж привязан к одной подписке, поэтому деньги не задваиваются. Если считать по людям, один платеж посчитается столько раз, сколько человек пользуется подпиской, и выручка выйдет больше настоящей (раздел 3). Тариф нужен, потому что подорожание у тарифов разное (+25% и +12,5%), а внутри одного тарифа цена платежа — это чистая цена. Канал — чтобы проверить гипотезу 4. Месяц — потому что подписка списывается раз в 30 дней, а дальше всё пересчитывается на день.

Период. База — январь–февраль 2026, последние полные месяцы до подорожания. После — март–июль 2026. Август не берем: он неполный, и 22–23 августа были массовые отмены partner_telecom, которые к цене отношения не имеют. Для оттока берем историю с сентября 2025, чтобы было с чем сравнивать.

Сегмент. Вся база. Мой сегмент из ДЗ-2 (4 653 клиента, 3 204 подписки) слишком маленький: изменение оттока на десятые доли процента на нём потерялось бы в шуме.

In [ ]:
PRICE_CHANGE = "2026-03-01"                 
BASE = ("2026-01-01", "2026-03-01")         
AFTER = ("2026-03-01", "2026-08-01")    
HISTORY = "2025-09-01"                      
DAYS_IN_MONTH = 365.25/12                 

LAST_MONTH = (pd.Timestamp(AFTER[1]) - pd.DateOffset(months=1)).strftime("%Y-%m-%d")


def period_of(month: str) -> str:
    if BASE[0][:7] <= month < BASE[1][:7]:
        return "база"
    if AFTER[0][:7] <= month < AFTER[1][:7]:
        return "после"
    return "история"


overview["period"] = overview["month"].map(period_of)
days = overview.groupby("period")["days_obs"].sum()          

print("база:", BASE, "| после:", AFTER, "| история с:", HISTORY)
print(days.to_string())

In [ ]:
def load_monthly(client_filter: str = "true", first: str = "2025-09-01", last: str = "2025-12-01") -> pd.DataFrame:
    """Месяц - тариф - канал: активные подписки, число платежей, выручка."""
    return q(f"""
        with seg as (
            select client_id, acquisition_channel as channel
            from prime.clients
            where {client_filter}
        ),
        act as (
            select to_char(m.month_start, 'YYYY-MM') as month, s.tariff_id, c.channel, count(*) as active
            from generate_series(cast(:first as timestamptz), cast(:last as timestamptz), interval '1 month') as m(month_start)
            join prime.subscriptions s
              on s.started_at < m.month_start + interval '1 month'
             and (s.ended_at is null or s.ended_at >= m.month_start)
            join seg c on c.client_id = s.owner_client_id
            group by 1, 2, 3
        ),
        rev as (
            select to_char(p.paid_at, 'YYYY-MM') as month, s.tariff_id, c.channel,
                   count(*) as n_payments, sum(p.amount) as revenue
            from prime.payments p
            join prime.subscriptions s on s.subscription_id = p.subscription_id
            join seg c on c.client_id = s.owner_client_id
            where p.status = 'success'
              and p.paid_at >= cast(:first as timestamptz)
              and p.paid_at < cast(:last as timestamptz) + interval '1 month'
            group by 1, 2, 3
        )
        select a.month, a.tariff_id, a.channel, a.active,
               coalesce(r.n_payments, 0) as n_payments, coalesce(r.revenue, 0) as revenue
        from act a
        left join rev r
          on r.month = a.month and r.tariff_id = a.tariff_id and r.channel is not distinct from a.channel
        order by 1, 2, 3
    """, first=first, last=last)


monthly = load_monthly("true", HISTORY, LAST_MONTH)
monthly_raw = monthly.copy()      

# Клиентов без канала не выбрасываем, оставляем отдельной строкой.
monthly["channel"] = monthly["channel"].fillna("без канала")
monthly["tariff"] = monthly["tariff_id"].map({1: "Базовый", 2: "Расширенный"})
monthly["period"] = monthly["month"].map(period_of)

print(len(monthly), "строк; месяцы:", monthly["month"].min(), "…", monthly["month"].max())
monthly.head(8)

In [ ]:
# типы колонок и пропуски смотрим 
print(monthly.dtypes.to_string())
print()
print("пропусков:", int(monthly.isna().sum().sum()))
print("пропусков канала в сырой копии:", int(monthly_raw["channel"].isna().sum()), "строк")
print("тарифов без названия:", int(monthly["tariff"].isna().sum()))

## 2. Метрики

- выручка в день = выручка месяца/дни с данными; 30,4 — выручка нормального месяца. Основная метрика, её можно сравнивать между месяцами разной длины.
- активные подписки — подписка началась до конца месяца и не закончилась до его начала. Первый множитель: сколько подписок есть.
- платежей на подписку в день = платежи/дни с данными/активные. Второй множитель: как часто подписки платят.
- цена платежа = выручка/платежи, отдельно по тарифу. Третий множитель — в нём должно быть видно подорожание (гипотеза 1).
- доля тарифа = активные подписки тарифа/все активные. Для гипотезы 2.
- отток = закончились в месяце/активные на начало месяца, без пробных подписок. Для гипотезы 3.
- доля отмен too_expensive среди отмен с указанной причиной. Тоже для гипотезы 3: если цена отпугивает, эта доля должна вырасти.

Внутри тарифа: выручка в день = активные * платежей на подписку в день * цена платежа. Вся выручка — сумма двух тарифов.

In [ ]:
m = (monthly.groupby(["month", "tariff"], as_index=False)[["active", "n_payments", "revenue"]].sum()
            .merge(overview[["month", "days_obs"]], on="month", how="left", validate="m:1"))

m["per_sub_day"] = m["n_payments"] / m["days_obs"] / m["active"]     
m["price"] = m["revenue"] / m["n_payments"]                          
m["revenue_per_day"] = m["revenue"] / m["days_obs"]
m["period"] = m["month"].map(period_of)

m.round(4)

In [ ]:
active = m.pivot_table(index="month", columns="tariff", values="active", aggfunc="sum")
share = active.div(active.sum(axis=1), axis=0) * 100

share.round(1)

## 3. Контроли

До выводов проверяются четыре вещи: не раздувается ли выручка из-за шеринга, сходится ли выручка monthly с прямой суммой платежей, не теряются ли клиенты без канала и какие цены реально списывались.

In [ ]:
share_check = q("""
    with pay as (
        select payment_id, subscription_id, amount
        from prime.payments
        where status = 'success'
          and paid_at >= cast(:a as date) and paid_at < cast(:b as date)
    ),
    users as (                -- все, кто пользуется подпиской: владелец + участники
        select subscription_id, owner_client_id as client_id from prime.subscriptions
        union all
        select subscription_id, client_id from prime.subscription_members
    )
    select (select count(*)                   from pay) as n_payments,
           (select count(distinct payment_id) from pay) as n_payment_ids,
           (select sum(amount)                from pay) as revenue,
           (select count(*)        from pay join prime.subscriptions s using (subscription_id)) as n_via_subs,
           (select sum(pay.amount) from pay join prime.subscriptions s using (subscription_id)) as revenue_via_subs,
           (select count(*)        from pay join users using (subscription_id)) as n_via_users,
           (select sum(pay.amount) from pay join users using (subscription_id)) as revenue_via_users
""", a=BASE[0], b=AFTER[1]).iloc[0]

inflation = share_check["revenue_via_users"] / share_check["revenue"]
sharing_ok = (share_check["n_payments"] == share_check["n_payment_ids"] == share_check["n_via_subs"]
              and abs(share_check["revenue_via_subs"] - share_check["revenue"]) < 0.01)

print(f"платежей {int(share_check['n_payments'])}, через подписку {int(share_check['n_via_subs'])},"
      f" через всех пользователей {int(share_check['n_via_users'])}")
print("выручка по платежам:       ", rub(share_check["revenue"]))
print("выручка «по пользователям»:", rub(share_check["revenue_via_users"]), f"— завышена в {inflation:.2f} раза")

In [ ]:
from_monthly = monthly.groupby("month")["revenue"].sum()
control = overview.set_index("month").loc[from_monthly.index, "revenue"]
money_gap = (from_monthly - control).abs().max()

lost = monthly_raw.pivot_table(index="channel", columns="month", values="revenue", aggfunc="sum").sum()
kept = monthly.pivot_table(index="channel", columns="month", values="revenue", aggfunc="sum").sum()
no_channel_lost = (control - lost).abs().max()
channels_gap = (control - kept).abs().max()

In [ ]:

tariffs = q("select * from prime.tariffs order by tariff_id, valid_from")
a = tariffs.reset_index(names="row_id")
pairs = a.merge(a, on="tariff_id", suffixes=("_a", "_b"))
overlap = pairs[(pairs["row_id_a"] < pairs["row_id_b"])
                & (pairs["valid_from_a"] <= pairs["valid_to_b"])
                & (pairs["valid_from_b"] <= pairs["valid_to_a"])]

amounts = q("""
    select s.tariff_id,
           case when p.paid_at < cast(:hike as date) then 'до' else 'после' end as period,
           p.amount, count(*) as n_payments
    from prime.payments p
    join prime.subscriptions s on s.subscription_id = p.subscription_id
    where p.status = 'success'
      and p.paid_at >= cast(:a as date) and p.paid_at < cast(:b as date)
    group by 1, 2, 3
    order by 1, 2 desc, 4 desc
""", hike=PRICE_CHANGE, a=BASE[0], b=AFTER[1])

def pairs_in(period: str) -> set:
    part = amounts[amounts["period"] == period]
    return set(zip(part["tariff_id"], part["amount"]))


prices_before, prices_after = pairs_in("до"), pairs_in("после")
prices_ok = prices_before == {(1, 199), (2, 399)} and prices_after == {(1, 249), (2, 449)}

print(tariffs[["tariff_id", "tariff_name", "monthly_fee", "valid_from", "valid_to"]].to_string(index=False))
print("\nпересекающихся пар интервалов:", len(overlap))
print()
print(amounts.to_string(index=False))

In [ ]:
print(f"шеринг платёж - подписка m:1, выручка без раздутия;"
      f" через всех пользователей было бы *{inflation:.2f}"
      f" - {'ok' if sharing_ok else 'ВНИМАНИЕ'}")
print(f"деньги monthly против прямой суммы платежей: расхождение {money_gap:.2f} руб"
      f" - {'ok' if money_gap < 0.01 else 'ВНИМАНИЕ'}")
print(f"каналы без fillna pivot_table потерял бы до {rub(no_channel_lost)} в месяц;"
      f" с «без канала» расхождение {channels_gap:.2f} руб"
      f" - {'ok' if channels_gap < 0.01 else 'ВНИМАНИЕ'}")
print(f"цены пересечений в справочнике {len(overlap)} — цену берём из платежей;"
      f" списывалось до {sorted(prices_before)}, после {sorted(prices_after)}"
      f" - {'ok' if prices_ok else 'ВНИМАНИЕ: есть суммы не по прайсу'}")

Что показали проверки. Двойного счёта нет: каждый платёж попадает в расчёт один раз. Если бы считалась выручка по всем пользователям подписки, она завысилась бы примерно в 2,77 раза. Выручка таблицы monthly сходится с прямой суммой платежей до копейки. Клиенты без канала не теряются: без отдельной строки «без канала» pivot_table потерял бы до 1,3 млн руб в месяц. В справочнике тарифов интервалы пересекаются, поэтому цена берется из самих платежей: до 1 марта у «Базового» списывалось только 199 руб, у «Расширенного» 399 руб, после — 249 и 449 руб.

## 4. Факторная декомпозиция

Сначала смотрим, где изменилась выручка (каналы и тарифы), потом раскладываем изменение на множители. Все суммы — в рублях за «нормальный месяц»: выручка периода/дни с данными * 30,4.

### 4.1 Где: каналы и тарифы

In [ ]:
level = monthly.groupby("period")["revenue"].sum()/days * DAYS_IN_MONTH
total_delta = level["после"] - level["база"]

naive = overview.groupby("period")["revenue"].mean()
naive_delta = naive["после"] - naive["база"]

print("выручка нормального месяца:", rub(level["база"]), "-", rub(level["после"]),
      f"({total_delta / level['база']:+.1%})")
print("наивный прирост:  ", rub(naive_delta))

In [ ]:
def where(by: str) -> pd.DataFrame:
    t = monthly.pivot_table(index=by, columns="period", values="revenue", aggfunc="sum")[["база", "после"]]
    t = t / days[["база", "после"]] * DAYS_IN_MONTH     
    t["delta"] = t["после"] - t["база"]
    assert abs(t["delta"].sum() - total_delta) < 0.01, f"{by}: части не сходятся с итогом"
    t = t.sort_values("delta")
    t.loc["итого"] = t.sum()
    return t


by_channel = where("channel")
by_tariff = where("tariff")

print("по каналам, рублей в нормальный месяц")
print(by_channel.round().astype(int).to_string())
print("\nпо тарифам, рублей в нормальный месяц")
print(by_tariff.round().astype(int).to_string())

**Вывод.** Выручка выросла во всех каналах, примерно на 24–30% у каждого. Больше всего в рублях дал organic, но просто потому, что это самый большой канал. Оба тарифа тоже выросли. Одного канала, который всё объясняет, нет — гипотеза 4 не подтвердилась. Рост общий, как и ожидаешь от изменения цены сразу для всех.

### 4.2 За счёт чего: активные * платежи * цена

Внутри каждого тарифа раскладываем изменение цепными подстановками в порядке активные - платежей на подписку - цена, потом складываю тарифы. Цена идёт последней, поэтому её вклад — (новая цена − старая) * фактические платежи после подорожания.

In [ ]:
from prime.factors import contrib


def factors(rows: pd.DataFrame) -> dict:
    by_month = rows.groupby("month")[["active", "n_payments", "revenue"]].sum()   
    n_days = overview.set_index("month").loc[by_month.index, "days_obs"].sum()
    active = by_month["active"].mean()
    return {"active": active,
            "per_sub_day": by_month["n_payments"].sum() / n_days / active,
            "price": by_month["revenue"].sum() / by_month["n_payments"].sum()}


FACTORS = ["active", "per_sub_day", "price"]
LABELS = {"active": "активные подписки", "per_sub_day": "платежей на подписку", "price": "цена платежа"}
TARIFFS = ["Базовый", "Расширенный"]

x0 = {t: factors(monthly[(monthly["tariff"] == t) & (monthly["period"] == "база")]) for t in TARIFFS}
x1 = {t: factors(monthly[(monthly["tariff"] == t) & (monthly["period"] == "после")]) for t in TARIFFS}

print(pd.DataFrame({(t, p): x[t] for t in TARIFFS for p, x in (("база", x0), ("после", x1))})
      .rename(index=LABELS).round(4).to_string())

In [ ]:
def tariff_effects(x0: dict, x1: dict, order: list[str]) -> pd.DataFrame:
    rows = {}
    for t in TARIFFS:
        rows[t] = contrib(x0[t], x1[t], order)
    return pd.DataFrame(rows).T * DAYS_IN_MONTH


effects_by_tariff = tariff_effects(x0, x1, FACTORS)
effects = effects_by_tariff.sum()      

assert abs(effects.sum() - total_delta) < 0.01, "сумма влияний не равна изменению выручки"

table = effects_by_tariff.rename(columns=LABELS)
table["всего"] = table.sum(axis=1)
table.loc["итого"] = table.sum()
print("рублей в нормальный месяц; январь–февраль - март–июль 2026; активные - платежи - цена")
print(table.round().astype(int).to_string())
print()
print(f"сумма влияний {rub(effects.sum())} — изменение выручки {rub(total_delta)}")
print(f"наивный прирост {rub(naive_delta)}; цене из него принадлежит {rub(effects['price'])}")

In [ ]:
rows = {}
for order in itertools.permutations(FACTORS):
    name = " - ".join(LABELS[f].split()[0] for f in order)
    rows[name] = tariff_effects(x0, x1, list(order)).sum()     
orders = pd.DataFrame(rows).T
orders.loc["среднее по порядкам"] = orders.mean()

orders.rename(columns=LABELS).round().astype(int)

In [ ]:
rows = {}
for month in sorted(monthly.loc[monthly["period"] == "после", "month"].unique()):
    x_month = {t: factors(monthly[(monthly["tariff"] == t) & (monthly["month"] == month)]) for t in TARIFFS}
    rows[month] = tariff_effects(x0, x_month, FACTORS).sum()
by_month = pd.DataFrame(rows).T.rename(columns=LABELS)
by_month["изменение"] = by_month.sum(axis=1)
by_month.round().astype(int)

Вывод. Из +13,09 млн руб в месяц цена дала +9,19 млн, рост числа активных подписок +3,92 млн, частота платежей почти не изменилась. При других порядках подстановок вклад цены 8,48–9,19 млн, так что вывод от порядка не зависит. Наивное сравнение «средний месяц после минус средний месяц до» даёт +14,06 млн и приписывает всё цене — это завышение почти на 5 млн в месяц.

Гипотеза 1 подтвердилась: рост в основном дала цена. Вклад активных подписок — это скорее обычный рост базы, который был и до марта, а не эффект подорожания.

### 4.3 Структура тарифов

Средняя цена платежа = сумма по тарифам (доля платежей × цена). Делю её рост на две части: сдвиг долей при старых ценах и рост цен при новых долях.

In [ ]:
payments = m.pivot_table(index="tariff", columns="period", values="n_payments", aggfunc="sum")
share0 = payments["база"] / payments["база"].sum()
share1 = payments["после"] / payments["после"].sum()
price0 = pd.Series({t: x0[t]["price"] for t in TARIFFS})
price1 = pd.Series({t: x1[t]["price"] for t in TARIFFS})

avg0, avg1 = (share0 * price0).sum(), (share1 * price1).sum()
by_structure = ((share1 - share0) * price0).sum()      
by_price = (share1 * (price1 - price0)).sum()          
assert abs(by_structure + by_price - (avg1 - avg0)) < 0.01, "сумма влияний не равна изменению"

print(pd.DataFrame({"доля платежей, база": share0, "доля платежей, после": share1,
                    "цена, база": price0, "цена, после": price1}).round(4).to_string())
print()
print(f"средняя цена платежа: {avg0:.2f} - {avg1:.2f} руб ({avg1 - avg0:+.2f} руб)")
print(f" за счёт структуры тарифов: {by_structure:+.2f} руб")
print(f" за счёт цен: {by_price:+.2f} руб")

Вывод. Доли тарифов не изменились: примерно 62% платежей у «Базового» и 38% у «Расширенного» и до, и после. Средняя цена платежа выросла на 50,08 руб, из них на сдвиг структуры приходится всего 0,08 руб, остальное — сами цены. Клиенты на «Расширенный» не переходили, гипотеза А2 не подтвердилась.

### 4.4 Отток

Отток по месяцам и тарифам — доля подписок, активных на начало месяца, которые закончились в этом месяце. Пробные подписки не берем. «Базовый» подорожал на 25%, «Расширенный» на 12,5%, поэтому смотрим, у какого тарифа отток вырос сильнее, и сравниваем разницу с погрешностью.

In [ ]:
subs = q("""
    select subscription_id, tariff_id, started_at, ended_at, cancel_reason
    from prime.subscriptions
    where not is_trial
    order by subscription_id
""")
subs["tariff"] = subs["tariff_id"].map({1: "Базовый", 2: "Расширенный"})


def churn_counts(subs: pd.DataFrame, start: str, end: str) -> tuple[int, int, int, int]:
    """Активные на start, сколько из них закончилось до end, у скольких есть причина,
    у скольких причина too_expensive."""
    start, end = pd.Timestamp(start, tz="UTC"), pd.Timestamp(end, tz="UTC")
    active = subs[(subs["started_at"] < start)
                  & (subs["ended_at"].isna() | (subs["ended_at"] >= start))]
    ended = active[active["ended_at"] < end]
    with_reason = int(ended["cancel_reason"].notna().sum())
    too_expensive = int((ended["cancel_reason"] == "too_expensive").sum())
    return len(active), len(ended), with_reason, too_expensive


month_starts = pd.date_range(HISTORY, AFTER[1], freq="MS").strftime("%Y-%m-%d")
rows = []
for start, end in zip(month_starts[:-1], month_starts[1:]):
    for tariff, g in subs.groupby("tariff"):
        active_start, ended, with_reason, too_expensive = churn_counts(g, start, end)
        rows.append({"month": start[:7], "tariff": tariff, "active_start": active_start, "ended": ended,
                     "with_reason": with_reason, "too_expensive": too_expensive})

churn = pd.DataFrame(rows)
churn["churn"] = churn["ended"] / churn["active_start"]
churn["period"] = np.where(churn["month"] < PRICE_CHANGE[:7], "до", "после")

churn.pivot_table(index="month", columns="tariff", values="churn", aggfunc="sum").mul(100).round(2)

In [ ]:
pooled = churn.groupby(["tariff", "period"], as_index=False)[["active_start", "ended", "with_reason", "too_expensive"]].sum()
pooled["churn"] = pooled["ended"] / pooled["active_start"]
pooled["share_too_expensive"] = pooled["too_expensive"] / pooled["with_reason"]
# Погрешность доли — стандартная ошибка √(p(1−p)/n).
pooled["se"] = np.sqrt(pooled["churn"] * (1 - pooled["churn"]) / pooled["active_start"])
print(pooled.round(4).to_string(index=False))
print()

change, change_se = {}, {}
for t in TARIFFS:
    before = pooled[(pooled["tariff"] == t) & (pooled["period"] == "до")].iloc[0]
    after = pooled[(pooled["tariff"] == t) & (pooled["period"] == "после")].iloc[0]
    change[t] = after["churn"] - before["churn"]                      
    change_se[t] = np.sqrt(before["se"] ** 2 + after["se"] ** 2)
    print(f"{t}: отток {change[t] * 100:+.2f} пп +/- {1.96 * change_se[t] * 100:.2f}")

did = change["Базовый"] - change["Расширенный"]
did_ci = 1.96 * np.sqrt(change_se["Базовый"] ** 2 + change_se["Расширенный"] ** 2)
print(f"\nразница приростов («Базовый» минус «Расширенный»): {did * 100:+.2f} пп +/- {did_ci * 100:.2f}")

Вывод. Отток около 3,9–4,0% в месяц и до, и после подорожания, у обоих тарифов. У «Базового» он изменился на −0,06 пп (+/- 0,08), у «Расширенного» на 0,00 пп (+/- 0,10), разница между ними −0,06 +/- 0,13 — в пределах погрешности. Если бы цена отпугивала, сильнее должен был отреагировать «Базовый» (+25%), но этого не видно. Доля отмен «слишком дорого» тоже не изменилась: 24,2% до и после. Отмен 5–6 тыс. в месяц, так что рост оттока даже на 0,1 пп был бы заметен. Гипотеза 3 не подтвердилась.

Оговорка: «Расширенный» тоже подорожал, поэтому это сравнение сильного подорожания со слабым, а не с контрольной группой без изменений.

## 5. Графики

In [ ]:
ACCENT, GRAY, GRID = "#C1272D", "#9CA3AF", "#E5E7EB"


def style(ax: plt.Axes, title: str, ylabel: str, caption: str) -> None:
    """Оформление графика: заголовок слева, подписи осей, без лишних рамок."""
    ax.set_title(title, loc="left")
    ax.set_ylabel(ylabel)
    ax.set_xlabel(caption, fontsize=9, color="#6B7280")
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", color=GRID)
    ax.set_axisbelow(True)

### График 1. Выручка в день по месяцам

In [ ]:
g = overview[overview["month"] >= HISTORY[:7]].copy()
g = g[g["month"] < AFTER[1][:7]]
pre = g.loc[g["month"] < PRICE_CHANGE[:7], "growth_per_day_pct"].mean()
post = g.loc[g["month"] > PRICE_CHANGE[:7], "growth_per_day_pct"].mean()
jump = g.loc[g["month"] == PRICE_CHANGE[:7], "growth_per_day_pct"].iloc[0]

fig, ax = plt.subplots(figsize=(10, 4.6), layout="constrained")
before_hike = g["month"] < PRICE_CHANGE[:7]
ax.plot(g["month"], g["revenue_per_day"] / 1e6, color=GRAY, marker="o", zorder=2)
ax.plot(g.loc[~before_hike, "month"], g.loc[~before_hike, "revenue_per_day"] / 1e6, color=ACCENT,
        marker="o", linewidth=2.5, zorder=3)
ax.axvline(before_hike.sum() - 0.5, color="#111827", linestyle="--", linewidth=0.8)
ax.annotate("подорожание 1 марта", (before_hike.sum() - 0.45, ax.get_ylim()[0]), fontsize=9,
            xytext=(0, 5), textcoords="offset points")
style(ax, f"В марте выручка в день выросла на {jump:.0f} %; до и после — примерно "
          f"{pre:+.1f} % и {post:+.1f} % в месяц",
      "млн ₽ в день",
      "Месяц по UTC. Выручка месяца/дни с данными: апрель без 14–15 числа. Источник: prime.payments, успешные платежи.")
plt.show()

### График 2. Из чего сложилось изменение выручки

Каскад изменений, а не уровней: при оси от нуля шаги в несколько миллионов на фоне 50 млн были бы почти не видны.

In [ ]:
from prime.factors import waterfall


steps = effects.rename({"active": "активные\nподписки", "per_sub_day": "платежей\nна подписку",
                        "price": "цена\nплатежа"}) / 1000
ax = waterfall(steps, title=f"Цена дала {effects['price'] / 1000:+,.0f} тыс. ₽ в месяц "
                            f"из {total_delta / 1000:+,.0f} тыс. прироста".replace(",", " "))
ax.set_ylabel("тыс. ₽ в нормальный месяц")
ax.set_xlabel("Январь–февраль - март июль 2026, выручка на день с данными * 30,4. "
              "Порядок подстановок — слева направо; тарифы сложены.", fontsize=9, color="#6B7280")
plt.show()

### График 3. Отток по тарифам

In [ ]:
wide = churn.pivot_table(index="month", columns="tariff", values="churn", aggfunc="sum") * 100
hike_x = (wide.index < PRICE_CHANGE[:7]).sum() - 0.5

fig, ax = plt.subplots(figsize=(10, 4.6), layout="constrained")
for t, color, shift in (("Расширенный", GRAY, 10), ("Базовый", ACCENT, -10)):
    ax.plot(wide.index, wide[t], color=color, marker="o", linewidth=2.5 if color == ACCENT else 1.5)
    ax.annotate(t, (len(wide) - 1, wide[t].iloc[-1]), xytext=(8, shift), textcoords="offset points",
                color=color, va="center")
ax.axvline(hike_x, color="#111827", linestyle="--", linewidth=0.8)
ax.set_ylim(bottom=0)
ax.set_xlim(right=len(wide) + 0.5)
style(ax, f"После подорожания отток «Базового» изменился на {change['Базовый'] * 100:+.1f} пп, "
          f"«Расширенного» — на {change['Расширенный'] * 100:+.1f} пп",
      "закончились в месяце, % от активных на начало",
      "Месяц по UTC, пробные подписки исключены. Пунктир — 1 марта 2026. Источник: prime.subscriptions.")
plt.show()

### График 4. Доля отмен «слишком дорого»

Над столбиками — сколько отмен с указанной причиной было в месяце.

In [ ]:
reasons = churn.groupby(["month", "period"], as_index=False)[["with_reason", "too_expensive"]].sum()
reasons["share"] = reasons["too_expensive"] / reasons["with_reason"] * 100
by_period = churn.groupby("period")[["too_expensive", "with_reason"]].sum()
pre_share = by_period.loc["до", "too_expensive"] / by_period.loc["до", "with_reason"]
post_share = by_period.loc["после", "too_expensive"] / by_period.loc["после", "with_reason"]

fig, ax = plt.subplots(figsize=(10, 4.6), layout="constrained")
bars = ax.bar(reasons["month"], reasons["share"],
              color=[ACCENT if p == "после" else GRAY for p in reasons["period"]], zorder=2)
ax.bar_label(bars, labels=[f"n={n}" for n in reasons["with_reason"]], padding=2, fontsize=8)
ax.margins(y=0.15)
style(ax, f"Доля отмен «слишком дорого»: {pre_share:.0%} до подорожания - {post_share:.0%} после",
      "too_expensive, % отмен с причиной",
      "Месяц отмены по UTC; красным — после 1 марта 2026. Источник: prime.subscriptions.cancel_reason.")
plt.show()

### График 5. Доля «Расширенного» среди активных подписок

In [ ]:
ext = share["Расширенный"]
ext = ext[ext.index < AFTER[1][:7]]

fig, ax = plt.subplots(figsize=(10, 4.6), layout="constrained")
ax.plot(ext.index, ext, color=ACCENT, marker="o", linewidth=2.5)
ax.axvline((ext.index < PRICE_CHANGE[:7]).sum() - 0.5, color="#111827", linestyle="--", linewidth=0.8)
ax.set_ylim(0, 100)      # шкала от 0 до 100 %, иначе сотые доли процента выглядят как скачок
style(ax, f"Доля «Расширенного» среди активных: {ext.iloc[0]:.1f} % - {ext.iloc[-1]:.1f} %",
      "% активных подписок",
      "Месяц по UTC. Пунктир — 1 марта 2026. Источник: prime.subscriptions.")
plt.show()

## 6. Вывод и рекомендация

Сводка чисел:

In [ ]:
print("выручка нормального месяца:", rub(level["база"]), "->", rub(level["после"]))
print("изменение:", rub(total_delta), "| наивное сравнение:", rub(naive_delta))
print("цена:", rub(effects["price"]), "| активные подписки:", rub(effects["active"]),
      "| платежей на подписку:", rub(effects["per_sub_day"]))
print("цена при разных порядках:", rub(orders["price"].iloc[:-1].min()), "…", rub(orders["price"].iloc[:-1].max()))
print(f"средняя цена платежа: структура {by_structure:+.2f} руб, цены {by_price:+.2f} ₽")
print(f"отток: «Базовый» {change['Базовый'] * 100:+.2f} пп, «Расширенный» {change['Расширенный'] * 100:+.2f} пп, "
      f"разница {did * 100:+.2f} +/- {did_ci * 100:.2f}")
print(f"доля too_expensive: {pre_share:.1%} - {post_share:.1%}")
print(f"раздутие выручки при счёте по пользователям: *{inflation:.2f}")

### Вывод

Что случилось. Выручка от подписок в «нормальный месяц» в марте–июле 2026 на 13,09 млн руб выше, чем в январе–феврале. До марта выручка в день росла ровно, примерно на 3% в месяц, в марте — скачок на 21%.

За счёт чего. Внутри каждого тарифа раскладывала изменение цепными подстановками (активные подписки - платежей на подписку - цена), потом складывала тарифы. Цена дала +9,19 млн руб в месяц, рост числа подписок +3,92 млн, частота платежей — около нуля. Клиенты не переходили между тарифами, отток и доля отмен «слишком дорого» не изменились. Простое сравнение месяцев завышает эффект цены почти на 5 млн.

Чего не знаем. Разложение показывает, за счёт чего вырос показатель, но не доказывает причину: группы без подорожания не было. После марта рост выручки в день замедлился с примерлёно 3% до примерно 2% в месяц. Отток не вырос, значит, возможная причина — меньше новых подписчиков (не проверялось).

Что делать. Цены сохранить: подорожание приносит около 9 млн руб в месяц, а отток не изменился. Следующим шагом проверить приток: число новых подписок и конверсию из пробного периода до и после марта. «Базовый» выдержал +25% без роста оттока, поэтому следующее повышение лучше проверить A/B-тестом на части новых клиентов, а не вводить сразу для всех.